In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ f:\mathbb{R} \to \mathbb{R}, \quad y = \tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} $$

$$ f:\mathbb{R^n} \to \mathbb{R^n}, \quad \mathbf{y} = \tanh(\mathbf{x}) = (\tanh(x_1), \tanh(x_2), \dots, \tanh(x_n)) $$

$$
\frac{df}{dx_i} = 1 - y_i^2
$$

$$
J_f(\mathbf{x}) =
\begin{bmatrix}
    \frac{\partial f}{\partial x_1} & 0 & \cdots & 0 \\
    0 & \frac{\partial f}{\partial x_2} & \cdots & 0 \\
    \vdots & \vdots & \ddots & \vdots \\
    0 & 0 & \cdots & \frac{\partial f}{\partial x_n}
\end{bmatrix}
$$

$$ df = J_f(\mathbf{x}) \cdot d\mathbf{x} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{df}, df \right \rangle =
\left \langle \frac{dF}{d\mathbf{x}}, d\mathbf{x} \right \rangle
$$

$$
\left \langle \frac{dF}{df}, J_f(\mathbf{x}) \cdot d\mathbf{x} \right \rangle =
\left \langle \frac{dF}{d\mathbf{x}}, d\mathbf{x} \right \rangle \implies
$$

$$ \frac{dF}{d\mathbf{x}} = J_f(\mathbf{x})^\top \, \frac{dF}{df} $$

$$ \frac{dF}{d\mathbf{x}} = \frac{df}{d\mathbf{x}} \odot \frac{dF}{df} $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import sigmoid # type: ignore
from approx import approx # type: ignore


def tanh(x: tr.Tensor) -> tr.Tensor:
    """Hyperbolic tangent function."""
    
    y = 2 * sigmoid.sigmoid(2 * x) - 1
    return y
    

class TanhFunction(autograd.Function):
    """Custom autograd function with `forward` and `backward` passes for the `tanh`."""

    @staticmethod
    def forward(ctx, x: tr.Tensor) -> tr.Tensor:
        y = tanh(x)  
        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, ]:
        (y, ) = ctx.saved_tensors

        df_dx = 1 -y ** 2
        dF_dx = df_dx * dF_df

        return (dF_dx, )
    

class Tanh(nn.Module):
    """Custom module for the `tanh`."""

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        y = TanhFunction.apply(x)
        return y


In [ ]:
# Unit tests

def test_basic() -> None:
    assert tanh(1000) == approx(tr.tensor(1.0))
    assert tanh(-1000) == approx(tr.tensor(-1.0))
    assert tanh(0) == approx(tr.tensor(0.0))
    assert tanh(1) == approx(tr.tensor(0.76159))
    assert tanh(-1) == approx(tr.tensor(-0.76159))


def test_gradcheck(samples) -> None:
    def func(x: tr.Tensor) -> tr.Tensor:
        return TanhFunction.apply(x)

    x = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (x, ), eps=0.001, atol=0.001)


def test_compare(samples) -> None:
    x = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = Tanh()(x1)
    F1 = y1.sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = tr.tanh(x2)
    F2 = y2.sum()
    F2.backward()

    assert y1 == approx(y2)
    assert x1.grad == approx(x2.grad)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)
    
    test_compare(1)
    test_compare(10)
    test_compare(100)